<a href="https://colab.research.google.com/github/Ethan-Brooke/APF-Paper-3-Ledgers-Entropy-Time-Cost/blob/main/APF_Reviewer_Walkthrough.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Paper 3 — Ledgers: Entropy, Time, and Cost
## Reviewer Walkthrough · Phase 22 Edition

[![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.18604844.svg)](https://doi.org/10.5281/zenodo.18604844)

**Author:** E.S. Brooke · **Paper:** v3.3 main + v1.2 supplement · **Codebase:** v6.9

---

### What this notebook is

Paper 3 is the **primary T_ACC formalisation receiver** and the **primary bridge-theorem receiver**. Entropy, arrow of time, CPTP dynamics, and the six-step bridge theorem proof all live here.

### What this notebook is **not**

A microscopic theory of decoherence (Paper 5), a derivation of $\Omega_\Lambda$ numerically (Paper 8), or a full cosmogenesis model (framework only).

### Before you begin

If you are a cold AI agent or human reviewer new to APF, read these four files in `ai_context/` **first**:

1. **`ARGUMENT_FLOW.md`** — the one-page structural spine.
2. **`LOCAL_VS_IMPORTED.md`** — what this paper proves vs imports.
3. **`CLAIMS_LEDGER.md`** — row-by-row attack surface.
4. **`DO_NOT_CLAIM.md`** — predictable overclaims and how to avoid them.


## §1 · Setup

Clone the paper-companion repo and load the rendering helpers.

In [ ]:
!git clone -q https://github.com/Ethan-Brooke/APF-Paper-3-Ledgers-Entropy-Time-Cost.git 2>/dev/null || true
%cd -q APF-Paper-3-Ledgers-Entropy-Time-Cost
!pip install -q -e . 2>&1 | tail -2

In [ ]:
from apf.bank import REGISTRY, get_check, run_all
from apf.apf_utils import dag_reset, dag_put, dag_has, dag_get
from fractions import Fraction
from IPython.display import display, Markdown, Latex, HTML
import inspect

print(f'Bank registry loaded: {len(REGISTRY)} checks.')
print('This repo: 12 bank-registered checks bundled.')

### Phase 22 `show()` helper

Every check returns a result dict. `show()` runs the check, badges the status colour-coded by epistemic tag, and surfaces the mathematical content inline.

Colour code: 🟢 `[P]` proved from A1 · 🟡 `[P_structural]` conditional on upstream · 🟣 `[P_arith]` arithmetic identity · 🔵 `[P+lattice]` lattice QCD input · 🟠 `[C]` conjecture · 🔴 `[FAIL]` check did not pass.

In [ ]:

def _epistemic_badge(tag):
    colors = {
        'P': ('🟢', '#2ecc71', 'proved from A1'),
        'P_structural': ('🟡', '#f1c40f', 'conditional on upstream derivation'),
        'P_arith': ('🟣', '#9b59b6', 'arithmetic identity once formula chosen'),
        'P+lattice': ('🔵', '#3498db', 'proved using lattice QCD input'),
        'C': ('🟠', '#e67e22', 'conjecture; open, flagged'),
    }
    emoji, col, explain = colors.get(str(tag).strip('[]'), ('⚪', '#7f8c8d', 'unknown'))
    return f'<span style="background:{col}22;border:1px solid {col};border-radius:4px;padding:2px 8px;color:{col};font-weight:600">{emoji} {tag}</span> <em style="color:#666;font-size:0.9em">{explain}</em>'


def _render_value(v):
    if isinstance(v, Fraction):
        return f'$\\displaystyle \\frac{{{v.numerator}}}{{{v.denominator}}} = {float(v):.6f}$'
    if isinstance(v, dict) and all(isinstance(val, Fraction) for val in v.values()):
        items = [f'{k}={frac.numerator}/{frac.denominator}' for k, frac in v.items()]
        return '$' + ',\\ '.join(items) + '$'
    if isinstance(v, float):
        return f'`{v:.9g}`'
    if isinstance(v, (list, tuple)) and len(v) < 8:
        return '`' + ', '.join(str(x) for x in v) + '`'
    return f'`{v}`'


def show(check_name, *, run=True, verbose=True):
    try:
        check = get_check(check_name)
    except KeyError:
        display(Markdown(f'**❓ Check `{check_name}` not found.**'))
        return None

    display(Markdown(f'#### `{check_name}`'))
    doc = (check.__doc__ or '').strip()
    first_line = doc.split('\n')[0] if doc else '(no docstring)'
    display(Markdown(f'**Statement:** {first_line}'))

    if not run:
        return None

    try:
        result = check()
        passed = True
    except Exception as e:
        result = {'error': f'{type(e).__name__}: {e}'}
        passed = False

    tag = 'P'
    if isinstance(result, dict):
        for k in ('epistemic_status', 'epistemic', 'tag'):
            if k in result:
                tag = result[k]
                break
    if not passed:
        display(Markdown(f'**Status:** <span style="color:#e74c3c;font-weight:700">🔴 [FAIL]</span>'))
    else:
        display(Markdown(f'**Status:** {_epistemic_badge(tag)}'))

    if isinstance(result, dict):
        if 'key_result' in result:
            display(Markdown(f'**Key result:** {_render_value(result["key_result"])}'))
        if 'error' in result:
            display(Markdown(f'**Error:** `{result["error"]}`'))

        if verbose:
            skip = {'key_result', 'name', 'epistemic', 'epistemic_status', 'tag',
                    'dependencies', 'cross_refs', 'error', 'artifacts', 'statement',
                    'identity', 'consistent'}
            extra = {k: v for k, v in result.items() if k not in skip}
            if extra:
                rows = []
                for k, v in list(extra.items())[:10]:
                    rows.append(f'| `{k}` | {_render_value(v)} |')
                if rows:
                    display(Markdown('**Fields surfaced by the check:**\n\n| Field | Value |\n|---|---|\n' + '\n'.join(rows)))

        if 'dependencies' in result and result['dependencies']:
            deps = result['dependencies']
            if isinstance(deps, (list, tuple)):
                deps_str = ' · '.join(f'`{d}`' for d in deps)
                display(Markdown(f'**Depends on:** {deps_str}'))

    return result


print('show() helper loaded. Phase 22 gorgeous-math rendering enabled.')


## §2 · Admissible dynamics are CPTP — $T_{\rm CPTP}$

**$T_{\rm CPTP}$.** Admissibility forces completely-positive trace-preserving dynamics (matches Stinespring-Choi-GKSL-Lindblad form).

In [ ]:
show('check_T_CPTP')

## §3 · Cost monotone under dynamics — $T_\kappa$

**$T_\kappa$.** Cost increases monotonically under admissible dynamics (second-law backbone, via PLEC's A2 argmin component).

In [ ]:
show('check_T_kappa')

## §4 · Entropy and the arrow of time

- **$T_{\rm entropy}$** — entropy = log of admissible microstate count; monotone under CPTP (Lieb-Ruskai backbone).
- **$T_{\rm second\_law}$** — cost irreversibility ⇒ arrow of time.
- **$T_{\rm CPT}$** — CPT structure preserved.

In [ ]:
show('check_T_entropy')

In [ ]:
show('check_L_irr')

## §5 · T_ACC formalisation (Supplement §7, primary receiver)

Paper 3 Supplement v1.2 §7 formalises the Admissibility-Capacity Ledger as a first-class object:

$$\mathbf{ACC}(K, d_{\rm eff}) \;=\; (K,\, \mathrm{ACC}),\qquad \mathrm{ACC} := K\,\ln d_{\rm eff}$$

**Six regime projections** $\pi_T, \pi_G, \pi_Q, \pi_F, \pi_C, \pi_A$ (Definition 7.1–7.3).

**Three canonical interface factories** `acc_SM`, `acc_horizon`, `acc_quantum`.

**Four consistency identities** $I_1, I_2, I_3, I_4$ (Theorems 7.4–7.7) + composed $T_{\rm ACC\_unification}$ (Theorem 7.8).

In [ ]:
show('check_T_ACC_unification')

## §6 · The bridge theorem (Supplement §8, primary bridge receiver)

Paper 3 Supplement v1.2 §8 houses the full six-step proof of $T_{\rm interface\_sector\_bridge}$:

$$\boxed{
\begin{aligned}
\text{The T12 interface partition } V_{61} = V_{\rm local} \oplus V_{\rm global} \\
\text{governs the } T_{\rm horizon\_reciprocity} \text{ sector decomposition.}
\end{aligned}
}$$

$|\text{Sector A}| = 60$, $|\text{Sector B}| = \dim V_{\rm global} = 42$, $d_{\rm eff} = 60 + 42 = 102$.

Three corollaries C1/C2/C3 establish dimension-functor framing; Lemma 8.1 identifies $V_{\rm global}$ with the horizon absorber.

In [ ]:
show('check_T_interface_sector_bridge')

## §7 · Cosmogenesis and falsifiers

Staged constraint relaxation framework: early universe has higher admissibility capacity, relaxes through phase transitions.

Two **proposed falsifiers** flagged in v3.0 (open, awaiting data):

- **F4 ringdown memory** — LIGO / LISA sensitivity.
- **F5 pre-saturation imprints** — Planck follow-up.

## §7 · Full bank pass

Run every check in this repo's bundled codebase subset.

In [ ]:
from apf.bank import run_all
from collections import Counter
results = run_all()
outcomes = Counter()
for name, res in results.items():
    if isinstance(res, dict) and 'error' in res:
        outcomes['ERROR'] += 1
    else:
        outcomes['PASS'] += 1
print(f'Total: {len(results)} checks')
for status, n in outcomes.most_common():
    print(f'  {status}: {n}')

### Where to go next

- **Paper PDF** — the main paper + Technical Supplement in this repo.
- **`ai_context/`** — the four audit-native files (ARGUMENT_FLOW, LOCAL_VS_IMPORTED, CLAIMS_LEDGER, DO_NOT_CLAIM).
- **[Canonical codebase v6.9](https://doi.org/10.5281/zenodo.18604548)** — the full 420-theorem bank.
- **[Paper 8 companion repo](https://github.com/Ethan-Brooke/APF-Paper-8-Admissibility-Capacity-Ledger)** — the pilot implementation of Phase 22 (anti-smuggling tests + minimal working example + full gorgeous-math Colab).

### Citation

```bibtex
@software{Brooke_Paper3_2026,
  author  = {Brooke, Ethan S.},
  title   = {Ledgers: Admissible Dynamics, Entropy, and the Arrow of Time},
  year    = 2026,
  version = {v3.3 main + v1.2 supplement},
  doi     = {10.5281/zenodo.18604844}
}
```

*Paper-companion repo · v3.3 main + v1.2 supplement · Phase 22 gorgeous-math edition · 2026-04-24.*